# 04. Submission Validator와 Train/Inference CLI 분리

> 작성일: 2026-07-07 KST
> 범위: DACON BARAM 2026 제출 CSV 검증기, CSV bytes/SHA256 연결, `train.py`와 `inference.py` 진입점 분리
> 안전 원칙: 이 노트북과 기본 CLI 실행은 실제 제출 CSV, 모델 파일, `outputs/` 산출물을 만들지 않는다.

이번 작업의 목적은 모델 성능을 올리는 것이 아니라 제출 운영에서 치명적인 실수를 막는 안전장치를 코드와 CLI 경계에 배치하는 것이다. DACON 제출은 파일 하나가 최종 결과를 대표하므로, 예측값이 좋아도 템플릿 정렬, 행 수, 인코딩, 해시 기록이 어긋나면 재현성과 제출 가능성이 동시에 무너진다.

이 노트북은 구현 설명서이자 실행 증거다. 각 셀은 synthetic DataFrame만 사용하며, 공식 `sample_submission.csv`는 스키마 확인용으로만 읽는다.

## 전문가 방법론

제출 validator는 모델링 코드 뒤에 붙는 부가 기능이 아니라, inference 저장 경로의 게이트로 설계한다. 운영 관점에서 검증 순서는 다음 기준을 따른다.

1. **템플릿 동일성 우선**: `forecast_id`, `forecast_kst_dtm`이 공식 `sample_submission`과 같은 순서로 완전히 일치해야 한다. 이 두 컬럼이 흔들리면 점수 산식 이전에 예측 대상이 바뀐다.
2. **구조 검증 후 값 검증**: 컬럼명과 순서, row count를 먼저 확인한 뒤 target NaN, 음수, capacity 초과를 확인한다. 구조가 틀린 파일에서 값 검증을 계속하면 오류 원인이 흐려진다.
3. **저장 bytes와 ledger hash의 단일 출처화**: CSV 저장 bytes는 `submission_to_csv_bytes()` 한 경로로만 만들고, 같은 bytes에서 SHA256을 계산한다. ledger에 기록한 해시와 파일 해시가 다르면 제출 파일을 다시 찾거나 재현할 수 없다.
4. **기본 실행은 무산출물**: CLI 기본 실행은 dry-run이다. 실제 모델 파일이나 제출 CSV는 명시적인 옵션이 있을 때만 만든다.
5. **TDD 적용**: 잘못된 submission 7종을 먼저 테스트로 작성하고 실패를 확인한 뒤 validator를 구현한다.

이 방식은 대회 제출에서 흔한 "파일은 생겼는데 제출할 수 없는 상태"를 줄이는 쪽에 초점을 둔다.

## 실패 사례 카탈로그

| 실패 사례 | 탐지 기준 | 위험 | validator 결정 |
|---|---:|---|---|
| 컬럼 순서 변경 | `['forecast_id', 'forecast_kst_dtm', kpx targets...]`와 정확히 비교 | DACON 파서 또는 내부 ledger가 다른 의미로 읽을 수 있음 | 실패 |
| row count 부족/초과 | 8,760행 고정 | 특정 시간대 누락 또는 중복 제출 | 실패 |
| `forecast_id` 불일치 | sample과 위치별 완전 일치 | 예측값이 다른 시간/대상에 매핑됨 | 실패 |
| `forecast_kst_dtm` 불일치 | sample과 위치별 완전 일치 | 시간축 shift 또는 formatting 오류 | 실패 |
| target NaN | target 3개 중 하나라도 결측 | 제출 파서 실패 또는 점수 손상 | 실패 |
| 음수 발전량 | target 값 `< 0` | 물리적으로 불가능하고 평가 안정성 저하 | 실패 |
| capacity 초과 | 그룹별 21.6MW/21.6MW/21.0MW 초과 | baseline clip 누락 또는 후처리 오류 | 실패 |
| 인코딩/해시 분리 | 파일 bytes와 ledger hash가 다른 경로에서 생성 | 재현성 ledger가 제출 파일을 증명하지 못함 | 실패 |

Decision Box까지는 점수 개선보다 제출 파일 무결성을 우선한다.

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
  PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
  sys.path.insert(0, str(SRC_DIR))

print(PROJECT_ROOT)

C:\Users\kik32\workspace\Dacon\2026-BARAM-Wind-Power-Prediction-AI-Competition


## 공식 sample_submission 스키마 확인

공식 sample은 검증의 기준점이다. 이 셀은 파일을 읽기만 하며, 어떤 제출 파일도 저장하지 않는다. 기대되는 row count는 윤년이 아닌 2025년의 시간별 예측 구간 8,760행이다.

In [2]:
sample_path = PROJECT_ROOT / "data" / "raw" / "open" / "sample_submission.csv"
sample_submission = pd.read_csv(sample_path)

schema_summary = {
  "rows": len(sample_submission),
  "columns": list(sample_submission.columns),
  "first_forecast_id": sample_submission.loc[0, "forecast_id"],
  "first_forecast_kst_dtm": sample_submission.loc[0, "forecast_kst_dtm"],
  "last_forecast_id": sample_submission.loc[len(sample_submission) - 1, "forecast_id"],
  "last_forecast_kst_dtm": sample_submission.loc[len(sample_submission) - 1, "forecast_kst_dtm"],
}
schema_summary

{'rows': 8760,
 'columns': ['forecast_id',
  'forecast_kst_dtm',
  'kpx_group_1',
  'kpx_group_2',
  'kpx_group_3'],
 'first_forecast_id': 'forecast_0001',
 'first_forecast_kst_dtm': '2025-01-01 01:00:00',
 'last_forecast_id': 'forecast_8760',
 'last_forecast_kst_dtm': '2026-01-01 00:00:00'}

## 새 validator API 로드

아래 API는 inference 저장 경로에서 재사용한다. `validate_submission()`은 report를 반환하고, `assert_valid_submission()`은 저장 전에 예외를 발생시킨다. `build_validated_submission_artifact()`는 검증을 통과한 DataFrame에 대해서만 CSV bytes와 SHA256을 만든다.

In [3]:
from baram.metrics import CAPACITY_KWH, TARGET_COLS
from baram.reproducibility import submission_to_csv_bytes
from baram.validation import (
  EXPECTED_SUBMISSION_ROWS,
  SUBMISSION_COLUMNS,
  build_validated_submission_artifact,
  validate_submission,
)

EXPECTED_SUBMISSION_ROWS, SUBMISSION_COLUMNS, CAPACITY_KWH

(8760,
 ['forecast_id',
  'forecast_kst_dtm',
  'kpx_group_1',
  'kpx_group_2',
  'kpx_group_3'],
 {'kpx_group_1': 21600, 'kpx_group_2': 21600, 'kpx_group_3': 21000})

## Synthetic 정상 submission 생성

공식 sample의 ID와 시간축을 그대로 복사하고 target만 안전한 양수 값으로 채운다. 실제 모델 예측이나 실제 제출 파일 저장은 하지 않는다.

In [4]:
valid_submission = sample_submission.copy()
valid_submission["kpx_group_1"] = 100.0
valid_submission["kpx_group_2"] = 200.0
valid_submission["kpx_group_3"] = 300.0

valid_report = validate_submission(valid_submission, sample_submission)
valid_report

SubmissionValidationReport(ok=True, errors=[], row_count=8760, columns=['forecast_id', 'forecast_kst_dtm', 'kpx_group_1', 'kpx_group_2', 'kpx_group_3'])

## 실패 사례 실행 결과

각 실패 사례는 한 가지 오류만 바꾼다. report의 `errors`를 보면 어떤 게이트에서 막혔는지 확인할 수 있다. 이 결과는 테스트와 같은 설계 의도를 공유한다.

In [5]:
def broken_cases(base):
  cases = {}

  wrong_columns = base[["forecast_kst_dtm", "forecast_id", *TARGET_COLS]].copy()
  cases["wrong_column_order"] = wrong_columns

  cases["wrong_row_count"] = base.iloc[:-1].copy()

  wrong_id = base.copy()
  wrong_id.loc[10, "forecast_id"] = "forecast_broken"
  cases["wrong_forecast_id"] = wrong_id

  wrong_time = base.copy()
  wrong_time.loc[10, "forecast_kst_dtm"] = "2025-01-03 03:00:00"
  cases["wrong_forecast_kst_dtm"] = wrong_time

  nan_target = base.copy()
  nan_target.loc[20, "kpx_group_1"] = pd.NA
  cases["target_nan"] = nan_target

  negative_target = base.copy()
  negative_target.loc[30, "kpx_group_2"] = -0.01
  cases["negative_target"] = negative_target

  capacity_exceeded = base.copy()
  capacity_exceeded.loc[40, "kpx_group_3"] = CAPACITY_KWH["kpx_group_3"] + 0.01
  cases["capacity_exceeded"] = capacity_exceeded

  return cases

case_rows = []
for case_name, candidate in broken_cases(valid_submission).items():
  report = validate_submission(candidate, sample_submission)
  case_rows.append({"case": case_name, "ok": report.ok, "errors": " | ".join(report.errors)})

pd.DataFrame(case_rows)

,case,ok,errors
0,wrong_column_order,False,column order mismatch: expected ['forecast_id'...
1,wrong_row_count,False,"row count mismatch: expected 8760, got 8759"
2,wrong_forecast_id,False,forecast_id mismatch at row 10
3,wrong_forecast_kst_dtm,False,forecast_kst_dtm mismatch at row 10
4,target_nan,False,kpx_group_1 NaN values: 1
5,negative_target,False,kpx_group_2 negative values: 1
6,capacity_exceeded,False,kpx_group_3 capacity exceeded values: 1


## CSV bytes와 SHA256 연결 확인

저장 전 artifact는 `submission_to_csv_bytes()`와 같은 bytes를 사용한다. `utf-8-sig`는 BOM을 포함하고, `utf-8`은 BOM이 없다. 둘 중 어떤 인코딩을 선택하더라도 ledger hash는 실제 저장될 bytes에서 계산되어야 한다.

In [6]:
artifact_sig = build_validated_submission_artifact(valid_submission, sample_submission, encoding="utf-8-sig")
artifact_plain = build_validated_submission_artifact(valid_submission, sample_submission, encoding="utf-8")

bytes_summary = pd.DataFrame([
  {
    "encoding": artifact_sig.encoding,
    "bytes": len(artifact_sig.csv_bytes),
    "has_bom": artifact_sig.csv_bytes.startswith(b"\xef\xbb\xbf"),
    "sha256": artifact_sig.sha256,
    "matches_repro_helper": artifact_sig.csv_bytes == submission_to_csv_bytes(valid_submission, encoding="utf-8-sig"),
    "ledger_hash_matches": artifact_sig.ledger_sha256 == hashlib.sha256(artifact_sig.csv_bytes).hexdigest(),
  },
  {
    "encoding": artifact_plain.encoding,
    "bytes": len(artifact_plain.csv_bytes),
    "has_bom": artifact_plain.csv_bytes.startswith(b"\xef\xbb\xbf"),
    "sha256": artifact_plain.sha256,
    "matches_repro_helper": artifact_plain.csv_bytes == submission_to_csv_bytes(valid_submission, encoding="utf-8"),
    "ledger_hash_matches": artifact_plain.ledger_sha256 == hashlib.sha256(artifact_plain.csv_bytes).hexdigest(),
  },
])
bytes_summary

,encoding,bytes,has_bom,sha256,matches_repro_helper,ledger_hash_matches
0,utf-8-sig,455588,True,c9287bbc57f9222cf46206987a681c9227e9b4c5c90735...,True,True
1,utf-8,455585,False,b370c198c3027dcac1b464be73c08215c695a6cdca72b8...,True,True


## CLI 분리 설계

`train.py`와 `inference.py`는 역할을 분리한다.

| 진입점 | 기본 실행 | 명시 실행 시 역할 | 저장 조건 |
|---|---|---|---|
| `python -m baram.train` | dry-run 안내만 출력 | train labels와 NWP 파일을 읽어 baseline bundle 학습 | `--run --model-output`이 있을 때만 모델 저장 |
| `python -m baram.inference` | dry-run 안내만 출력 | 모델 bundle과 test NWP로 submission 후보 생성 | validator 통과 후 `submission_to_csv_bytes()` bytes만 저장 |

이 분리는 제출 운영에서 가장 중요한 실패 모드 하나를 막는다. inference가 임의의 CSV writer로 곧장 저장하지 못하게 하고, validator와 canonical bytes 생성기를 반드시 통과하게 한다.

In [7]:
from baram.train import main as train_main
from baram.inference import main as inference_main

# ?? ??? ???? ??? ?? dry-run??.
train_status = train_main([])
inference_status = inference_main([])
{"train_status": train_status, "inference_status": inference_status}

BARAM train dry-run: no model file or outputs will be created.
BARAM inference dry-run: no submission CSV or outputs will be created.


{'train_status': 0, 'inference_status': 0}

## 마크다운 무결성 점검

이전 노트북 일부는 한국어 마크다운이 물음표 세 개로 치환된 흔적이 있다. 아래 셀은 기존 노트북의 의심 패턴을 읽기 전용으로 점검한다. 이번 04 노트북은 UTF-8 JSON으로 저장하고, 최종 검증에서 replacement 문자와 과도한 물음표 치환을 별도로 확인한다.

In [8]:
scan_rows = []
replacement_char = chr(0xFFFD)
for notebook_file in sorted((PROJECT_ROOT / "notebooks").glob("0*.ipynb")):
  notebook = json.loads(notebook_file.read_text(encoding="utf-8"))
  markdown_parts = [
    "".join(cell.get("source", []))
    for cell in notebook.get("cells", [])
    if cell.get("cell_type") == "markdown"
  ]
  markdown_text = "".join(markdown_parts)
  scan_rows.append(
    {
      "notebook": notebook_file.name,
      "replacement_char_count": markdown_text.count(replacement_char),
      "triple_question_count": markdown_text.count("???"),
      "markdown_cells": sum(1 for cell in notebook.get("cells", []) if cell.get("cell_type") == "markdown"),
    }
  )

pd.DataFrame(scan_rows)

,notebook,replacement_char_count,triple_question_count,markdown_cells
0,00_official_data_audit.ipynb,0,0,24
1,01_metric_reproduction.ipynb,0,0,33
2,02_official_baseline_reproduction.ipynb,0,0,19
3,03_reproducible_submission_ledger.ipynb,0,258,9
4,04_submission_validator_and_cli.ipynb,0,0,12


## Decision Box

**결정: validator를 inference 저장 경로의 필수 게이트로 둔다.**

- 제출 CSV 구조는 `sample_submission`을 기준으로 고정한다.
- row count는 8,760행으로 고정한다.
- `forecast_id`, `forecast_kst_dtm`은 sample과 위치별로 완전히 일치해야 한다.
- target 3개는 NaN, 음수, capacity 초과를 허용하지 않는다.
- CSV bytes는 `submission_to_csv_bytes()`로만 만들고, 같은 bytes에서 SHA256과 ledger hash를 계산한다.
- train CLI와 inference CLI는 분리하되 기본 실행은 dry-run으로 유지한다.

이 결정은 모델 개선 속도를 조금 늦추더라도 제출 파일 복구 가능성과 운영 안정성을 높인다.

## 다음 판단

다음 작업에서 판단할 항목은 성능 개선이 아니라 제출 운영 완성도다.

1. 실제 모델 저장 포맷은 `pickle`로 유지할지, metadata JSON을 별도 sidecar로 둘지 결정한다.
2. local validation split 결과와 submission ledger를 하나의 run registry로 묶을지 결정한다.
3. 최종 제출 전에는 `utf-8-sig`를 기본으로 쓰되 DACON 업로드 UI에서 plain `utf-8`도 허용되는지 수동 확인한다.
4. 이전 노트북의 깨진 마크다운 복구는 별도 문서 복구 작업으로 분리한다. 이번 작업에서는 새 노트북이 깨지지 않는 것을 검증한다.